# **R/S BENCHMARK — Neural Network DATASET GENERATION (from the PCE, not the emulator)**

This notebook loads the `pce_metamodel` files, one per time step.

## **1. Libraries**

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Config**

Must match [`02_train_pce.ipynb`](02_train_pce.ipynb) — `r_mean`/`r_std`/`s_mean`/`s_std` rebuild
the same `joint`, `times` must be the same grid, and `n_latent_samples` names the `pce_metamodel`
files being loaded below (it plays no role in this notebook beyond that — no latent sampling
happens here).

`n_points` is new: how many fresh $(R, S)$ points to query per time step. Since a PCE evaluation is
cheap, this can be far denser than the design sample count used to fit the PCEs themselves.

In [2]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 2500   # must match stage 1/2 — it names the pce_metamodel files
n_lambdas        = 4
n_points         = 5000   # (R, S) query points drawn per time step for the NN dataset

lambda3_fixed = 0.142157   # written by hand — same values as 02_train_pce_plot.ipynb, the PCE's own lambda 3 / lambda 4 are unreliable
lambda4_fixed = 0.131525

times = np.linspace(0, 100, 5, endpoint=True)  # must match stage 1/2
times

array([  0.,  25.,  50.,  75., 100.])

## 3. Rebuild the joint distribution

In [3]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## 4. Load the per-time-step PCE metamodels

One `pce_metamodel` per entry of `times`, as saved by `train_and_validate_pce_from_dataset_benchmark`
in stage 2.

In [4]:
pce_models = []
for t in times:
    with open(f'{n_latent_samples}_pce_metamodel_{t}_benchmark.pkl', 'rb') as f:
        pce_models.append(dill.load(f))

print(f"Loaded {len(pce_models)} PCE metamodels")

Loaded 5 PCE metamodels


## 5. Query the PCEs and stack the dataset

`generate_nn_dataset_benchmark` draws `n_points` fresh $(R, S)$ samples per time step, evaluates the
matching PCE on them, and stacks every time step into one dataframe with an explicit
`Time (years)` column. `lambda 1`/`lambda 2` come from the PCE prediction; `lambda 3`/`lambda 4` are
overridden with the hand-written `lambda3_fixed`/`lambda4_fixed` from section 2, since the PCE's own
fit for those two is unreliable.

In [5]:
print("="*60)
print("GENERATING THE NN DATASET FROM THE PCE MODELS")
print("="*60)

result = generate_nn_dataset_benchmark(
                                          pce_metamodels=pce_models,
                                          times=times,
                                          joint=joint,
                                          n_points=n_points,
                                          n_lambdas=n_lambdas,
                                          lambda3_fixed=lambda3_fixed,
                                          lambda4_fixed=lambda4_fixed,
                                          n_latent_samples=n_latent_samples,
                                          output_dir='.',
                                       )

df_nn = result['dataset_nn']
print(f"\nTotal rows: {len(df_nn)}")
df_nn.head()

GENERATING THE NN DATASET FROM THE PCE MODELS

----------------------------------------
GENERATING NN DATASET FROM 5 PCE MODELS
----------------------------------------
  lambda 3 fixed at 0.1422
  lambda 4 fixed at 0.1315
  t = 0.00 years: 5000 points queried from the PCE
  t = 25.00 years: 5000 points queried from the PCE
  t = 50.00 years: 5000 points queried from the PCE
  t = 75.00 years: 5000 points queried from the PCE
  t = 100.00 years: 5000 points queried from the PCE
The NN dataset has been saved!

Total rows: 25000


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.206907,1.854493,0.0,3.354403,6.357913,0.142157,0.131525
1,5.286790,2.431395,0.0,2.858020,5.292953,0.142157,0.131525
2,5.269668,2.067211,0.0,3.204697,5.916525,0.142157,0.131525
3,5.194718,2.683213,0.0,2.514461,4.938048,0.142157,0.131525
4,5.579349,0.870576,0.0,4.708886,8.263479,0.142157,0.131525


## 6. Sanity check

In [6]:
df_nn[['r', 's', 'Time (years)', 'lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']].describe()

,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,2.500000e+04,2.500000e+04
mean,4.999209,2.001523,50.000000,1.748215,6.975713,1.421570e-01,1.315250e-01
std,0.800410,0.598141,35.356046,1.224968,1.897423,2.775613e-17,5.551226e-17
min,1.621442,-0.413600,0.000000,-1.883904,3.289198,1.421570e-01,1.315250e-01
25%,4.458011,1.597986,25.000000,0.839671,5.714775,1.421570e-01,1.315250e-01
50%,4.999629,2.005805,50.000000,1.688220,6.577426,1.421570e-01,1.315250e-01
75%,5.540358,2.397406,75.000000,2.602289,7.795376,1.421570e-01,1.315250e-01
max,8.530733,4.702768,100.000000,6.205885,25.467624,1.421570e-01,1.315250e-01


## 5. Sanity check — GLD validity

The PCE is queried at fresh `(R, S)` points, so nothing guarantees that the four predicted
parameters describe a valid GLD. The binding constraint is $\lambda_2 > 0$: the scale parameter
of the FKML parameterization must be strictly positive, otherwise the quantile function is
inverted and the "distribution" is not one.

Rows violating it are dropped before the dataset is saved — a handful of invalid targets would
otherwise enter the neural-network loss as extreme outliers.

> With the degradation factor clamped at `t_final` this check is expected to remove **zero** rows.
> If it removes any, the offending region of the input space is reported rather than silently
> discarded: a non-empty removal means the PCE is being extrapolated somewhere it should not be.

In [7]:
n_before = len(df_nn)
invalid  = df_nn['lambda 2'] <= 0

print(f'rows before : {n_before}')
print(f'invalid     : {invalid.sum()} ({invalid.mean() * 100:.3f}%)')

if invalid.any():
    print('\nregion of the input space where the PCE produced an invalid GLD:')
    print(df_nn.loc[invalid, ['r', 's', 'Time (years)', 'lambda 1', 'lambda 2']].describe().loc[
              ['min', 'max']].round(3).to_string())
    df_nn = df_nn.loc[~invalid].reset_index(drop=True)

print(f'\nrows after  : {len(df_nn)}')
assert (df_nn['lambda 2'] > 0).all(), 'lambda 2 must be strictly positive'
df_nn.describe()

rows before : 25000
invalid     : 0 (0.000%)

rows after  : 25000


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,2.500000e+04,2.500000e+04
mean,4.999209,2.001523,50.000000,1.748215,6.975713,1.421570e-01,1.315250e-01
std,0.800410,0.598141,35.356046,1.224968,1.897423,2.775613e-17,5.551226e-17
min,1.621442,-0.413600,0.000000,-1.883904,3.289198,1.421570e-01,1.315250e-01
25%,4.458011,1.597986,25.000000,0.839671,5.714775,1.421570e-01,1.315250e-01
50%,4.999629,2.005805,50.000000,1.688220,6.577426,1.421570e-01,1.315250e-01
75%,5.540358,2.397406,75.000000,2.602289,7.795376,1.421570e-01,1.315250e-01
max,8.530733,4.702768,100.000000,6.205885,25.467624,1.421570e-01,1.315250e-01


In [8]:
# re-save the validated dataset under the same name the training notebook reads
with open(f'{n_latent_samples}_dataset_nn_benchmark.pkl', 'wb') as f:
    dill.dump(df_nn, f)

print(f'saved {len(df_nn)} validated rows to {n_latent_samples}_dataset_nn_benchmark.pkl')

saved 25000 validated rows to 2500_dataset_nn_benchmark.pkl
